# Hey Banco — Datathon 2026
## Tarea 0: Carga, inspección e integridad de los 4 datasets

| Dataset | Archivo | Filas | Cols |
|---------|---------|------:|-----:|
| Clientes | `hey_clientes.csv` | 15,025 | 22 |
| Productos | `hey_productos.csv` | 38,909 | 13 |
| Transacciones | `hey_transacciones.csv` | 802,384 | 22 |
| Conversaciones | `dataset_50k_anonymized.parquet` | 49,999 | 6 |

**Relaciones entre tablas:**
```
hey_clientes.user_id ──┬──► hey_productos.user_id
                       ├──► hey_transacciones.user_id
                       └──► dataset_50k_anonymized.user_id

hey_productos.producto_id ──► hey_transacciones.producto_id
```

## 0. Setup

In [14]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.4f}'.format)
pd.set_option('display.max_colwidth', 60)

BASE_TXN  = Path(r"Datathon_Hey_2026_dataset_transacciones 1/dataset_transacciones")
BASE_CONV = Path(r"Datathon_Hey_dataset_conversaciones 1/dataset_conversaciones")

print("Rutas configuradas:", BASE_TXN.resolve(), "|", BASE_CONV.resolve())

Rutas configuradas: C:\Users\Fernando\Documents\GitHub\Datathon-2026\Datathon_Hey_2026_dataset_transacciones 1\dataset_transacciones | C:\Users\Fernando\Documents\GitHub\Datathon-2026\Datathon_Hey_dataset_conversaciones 1\dataset_conversaciones


## 1. Carga de los 4 datasets

In [15]:
# ── hey_clientes ──────────────────────────────────────────────────────────────
df_clientes = pd.read_csv(
    BASE_TXN / "hey_clientes.csv",
    dtype={
        "user_id": str,
        "sexo": str,
        "estado": str,
        "ciudad": str,
        "nivel_educativo": str,
        "ocupacion": str,
        "canal_apertura": str,
        "preferencia_canal": str,
        "idioma_preferido": str,
    },
)

# ── hey_productos ─────────────────────────────────────────────────────────────
df_productos = pd.read_csv(
    BASE_TXN / "hey_productos.csv",
    dtype={"producto_id": str, "user_id": str, "tipo_producto": str, "estatus": str},
    parse_dates=["fecha_apertura", "fecha_ultimo_movimiento"],
)
# quitar columna de metadato si existe
if "es_dato_sintetico" in df_productos.columns:
    df_productos = df_productos.drop(columns=["es_dato_sintetico"])

# ── hey_transacciones ─────────────────────────────────────────────────────────
df_transacc = pd.read_csv(
    BASE_TXN / "hey_transacciones.csv",
    dtype={
        "transaccion_id": str,
        "user_id": str,
        "producto_id": str,
        "tipo_operacion": str,
        "canal": str,
        "categoria_mcc": str,
        "ciudad_transaccion": str,
        "estatus": str,
        "motivo_no_procesada": str,
        "dia_semana": str,
        "dispositivo": str,
    },
    parse_dates=["fecha_hora"],
)
if "es_dato_sintetico" in df_transacc.columns:
    df_transacc = df_transacc.drop(columns=["es_dato_sintetico"])

# ── dataset_50k conversaciones ────────────────────────────────────────────────
df_convs = pd.read_parquet(
    BASE_CONV / "dataset_50k_anonymized.parquet",
)
df_convs["user_id"] = df_convs["user_id"].astype(str)
df_convs["date"]    = pd.to_datetime(df_convs["date"], format="mixed")
df_convs["channel_source"] = df_convs["channel_source"].astype(str)

print("Todos los datasets cargados correctamente.")

ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.

## 2. Shapes

In [ ]:
shapes = pd.DataFrame(
    [
        {"Dataset": "hey_clientes",              "Filas": df_clientes.shape[0],  "Columnas": df_clientes.shape[1]},
        {"Dataset": "hey_productos",             "Filas": df_productos.shape[0], "Columnas": df_productos.shape[1]},
        {"Dataset": "hey_transacciones",         "Filas": df_transacc.shape[0],  "Columnas": df_transacc.shape[1]},
        {"Dataset": "dataset_50k_anonymized",    "Filas": df_convs.shape[0],     "Columnas": df_convs.shape[1]},
    ]
)
shapes["Filas"] = shapes["Filas"].map("{:,}".format)
display(shapes)

,Dataset,Filas,Columnas
0,hey_clientes,"15,025",22
1,hey_productos,"38,909",12
2,hey_transacciones,"802,384",21
3,dataset_50k_anonymized,"49,999",6


## 3. Tipos de datos

In [ ]:
for nombre, df in [
    ("hey_clientes", df_clientes),
    ("hey_productos", df_productos),
    ("hey_transacciones", df_transacc),
    ("dataset_50k_anonymized", df_convs),
]:
    print(f"\n=== {nombre} ===")
    display(
        df.dtypes.rename("dtype")
          .to_frame()
          .assign(ejemplo=lambda d: d.index.map(lambda c: str(df[c].iloc[0])))
    )


=== hey_clientes ===


,dtype,ejemplo
user_id,object,USR-00001
edad,int64,21
sexo,object,M
estado,object,Ciudad de México
ciudad,object,CDMX - Benito Juárez
nivel_educativo,object,Preparatoria
ocupacion,object,Empleado
ingreso_mensual_mxn,int64,24500
antiguedad_dias,int64,1554
es_hey_pro,bool,True



=== hey_productos ===


,dtype,ejemplo
producto_id,object,PRD-00000001
user_id,object,USR-00001
tipo_producto,object,cuenta_debito
fecha_apertura,datetime64[ns],2023-06-26 00:00:00
estatus,object,activo
limite_credito,float64,nan
saldo_actual,float64,80954.6
utilizacion_pct,float64,nan
tasa_interes_anual,float64,nan
plazo_meses,float64,nan



=== hey_transacciones ===


,dtype,ejemplo
transaccion_id,object,TXN-0000000055
user_id,object,USR-00001
producto_id,object,PRD-00000002
fecha_hora,datetime64[ns],2025-01-15 14:17:42
tipo_operacion,object,compra
canal,object,app_ios
monto,float64,33.88
comercio_nombre,object,DivertidoPark
categoria_mcc,object,entretenimiento
ciudad_transaccion,object,"Nueva York, NY"



=== dataset_50k_anonymized ===


,dtype,ejemplo
input,object,"Me enteré de una promo ""Supercashback Pagos Gubernamenta..."
output,object,Claro que puedo ayudarte! Para participar en la promoció...
date,datetime64[ns],2025-08-17 00:00:00
conv_id,object,000502c2-288c-41f6-b751-a8b45a376a81
user_id,object,USR-09092
channel_source,object,1


## 4. Valores nulos por columna

In [ ]:
def resumen_nulos(df: pd.DataFrame, nombre: str) -> pd.DataFrame:
    total = df.isnull().sum()
    pct   = (total / len(df) * 100).round(2)
    res   = pd.DataFrame({"nulos": total, "pct_%": pct})
    n_con_nulos = (res["nulos"] > 0).sum()
    print(f"=== {nombre}: {n_con_nulos}/{len(res)} columnas con nulos ===")
    if n_con_nulos == 0:
        print("  Sin nulos.")
        return res
    return res[res["nulos"] > 0].sort_values("pct_%", ascending=False)

display(resumen_nulos(df_clientes,  "hey_clientes"))
display(resumen_nulos(df_productos, "hey_productos"))
display(resumen_nulos(df_transacc,  "hey_transacciones"))
resumen_nulos(df_convs,             "dataset_50k_anonymized")

=== hey_clientes: 3/22 columnas con nulos ===


,nulos,pct_%
satisfaccion_1_10,751,5.0000
estado,432,2.8800
ciudad,432,2.8800


=== hey_productos: 6/12 columnas con nulos ===


,nulos,pct_%
plazo_meses,34359,88.3100
monto_mensualidad,34359,88.3100
limite_credito,24592,63.2000
utilizacion_pct,24592,63.2000
tasa_interes_anual,20118,51.7100
saldo_actual,3750,9.6400


=== hey_transacciones: 7/21 columnas con nulos ===


,nulos,pct_%
meses_diferidos,785071,97.8400
motivo_no_procesada,775775,96.6800
cashback_generado,621746,77.4900
comercio_nombre,370538,46.1800
dispositivo,202063,25.1800
descripcion_libre,61752,7.7000
ciudad_transaccion,19267,2.4000


=== dataset_50k_anonymized: 0/6 columnas con nulos ===
  Sin nulos.


,nulos,pct_%
input,0,0.0000
output,0,0.0000
date,0,0.0000
conv_id,0,0.0000
user_id,0,0.0000
channel_source,0,0.0000


> **Nota:** Los nulos en `hey_productos` (límite_crédito, utilizacion_pct, plazo, mensualidad) y en `hey_transacciones` (motivo_no_procesada, meses_diferidos, cashback_generado) son **nulos estructurales** — solo aplican a ciertos tipos de producto u operación, no son datos faltantes por error.

## 5. Estadísticas descriptivas

In [ ]:
print("=== hey_clientes — numéricas ===")
display(df_clientes.describe())

=== hey_clientes — numéricas ===


,edad,ingreso_mensual_mxn,antiguedad_dias,score_buro,dias_desde_ultimo_login,satisfaccion_1_10,num_productos_activos
count,"15,025.0000","15,025.0000","15,025.0000","15,025.0000","15,025.0000","14,274.0000","15,025.0000"
mean,37.1878,"29,793.6772",935.1071,618.4516,18.7688,7.4753,2.2328
std,10.1899,"19,171.1565",512.9585,127.1600,35.3056,1.8782,1.1433
min,18.0000,"4,500.0000",7.0000,295.0000,0.0000,3.0000,0.0000
25%,29.0000,"16,000.0000",490.0000,529.0000,2.0000,6.0000,1.0000
50%,36.0000,"24,000.0000",942.0000,631.0000,7.0000,8.0000,2.0000
75%,45.0000,"37,000.0000","1,375.0000",719.0000,15.0000,9.0000,3.0000
max,60.0000,"99,000.0000","1,825.0000",850.0000,180.0000,10.0000,5.0000


In [ ]:
print("=== hey_clientes — categóricas ===")
for col in df_clientes.select_dtypes(include=["object", "bool"]).columns:
    vc = df_clientes[col].value_counts(dropna=False)
    print(f"\n{col}  ({vc.shape[0]} únicos):")
    display(vc.head(8))

=== hey_clientes — categóricas ===

user_id  (15025 únicos):


user_id
USR-00001    1
USR-10022    1
USR-10010    1
USR-10011    1
USR-10012    1
USR-10013    1
USR-10014    1
USR-10015    1
Name: count, dtype: int64


sexo  (3 únicos):


sexo
M     7281
H     7157
SE     587
Name: count, dtype: int64


estado  (18 únicos):


...

idioma_preferido
es_MX    14544
en_US      481
Name: count, dtype: int64


tiene_seguro  (2 únicos):


tiene_seguro
False    10883
True      4142
Name: count, dtype: int64


patron_uso_atipico  (2 únicos):


patron_uso_atipico
False    14262
True       763
Name: count, dtype: int64

In [ ]:
print("=== hey_productos ===")
display(df_productos.describe())
print("\ntipo_producto:")
display(df_productos["tipo_producto"].value_counts())
print("\nestatus:")
display(df_productos["estatus"].value_counts())

=== hey_productos ===


,fecha_apertura,limite_credito,saldo_actual,utilizacion_pct,tasa_interes_anual,plazo_meses,monto_mensualidad,fecha_ultimo_movimiento
count,38909,"14,317.0000","35,159.0000","14,317.0000","18,791.0000","4,550.0000","4,550.0000",38909
mean,2024-08-17 04:30:16.793029632,"89,630.9283","60,007.0574",0.4328,29.1163,23.0677,"1,990.7032",2025-10-13 19:58:53.012927232
min,2020-12-21 00:00:00,"1,000.0000",100.9400,0.0501,3.5000,6.0000,36.2900,2025-08-30 00:00:00
25%,2023-12-17 00:00:00,"29,000.0000","15,596.2400",0.2361,14.6550,12.0000,738.2375,2025-09-21 00:00:00
50%,2024-12-07 00:00:00,"64,000.0000","36,290.0900",0.3700,31.7400,18.0000,"1,477.3150",2025-10-14 00:00:00
75%,2025-07-15 00:00:00,"111,000.0000","66,193.2650",0.5762,41.0600,24.0000,"2,784.1050",2025-11-06 00:00:00
max,2025-11-27 00:00:00,"494,000.0000","499,905.5700",1.0000,66.9700,60.0000,"11,379.4200",2025-11-28 00:00:00
std,NaN,"94,668.4970","84,090.7280",0.2504,15.4839,13.7514,"1,683.1258",NaN



tipo_producto:


tipo_producto
cuenta_debito                  15025
tarjeta_credito_hey             7565
inversion_hey                   4474
seguro_vida                     2480
credito_nomina                  2044
credito_personal                1549
cuenta_negocios                 1343
seguro_compras                  1270
tarjeta_credito_negocios        1111
tarjeta_credito_garantizada     1091
credito_auto                     957
Name: count, dtype: int64


estatus:


estatus
activo               33548
cancelado             2917
suspendido            1426
revision_de_pagos     1018
Name: count, dtype: int64

In [ ]:
print("=== hey_transacciones ===")
display(df_transacc.describe())
print("\ntipo_operacion:")
display(df_transacc["tipo_operacion"].value_counts())
print("\nestatus:")
display(df_transacc["estatus"].value_counts())
print(f"\nRango de fechas: {df_transacc['fecha_hora'].min()}  ->  {df_transacc['fecha_hora'].max()}")

=== hey_transacciones ===


,fecha_hora,monto,intento_numero,meses_diferidos,cashback_generado,hora_del_dia
count,802384,"802,384.0000","802,384.0000","17,313.0000","180,638.0000","802,384.0000"
mean,2025-06-18 10:12:00.894490112,"6,108.4960",1.0331,10.7846,14.7853,11.4427
min,2025-01-01 08:00:09,5.0100,1.0000,3.0000,0.1200,0.0000
25%,2025-03-29 02:42:13.500000,540.0000,1.0000,6.0000,3.5600,6.0000
50%,2025-06-18 01:01:14.500000,"1,740.0000",1.0000,9.0000,6.9300,11.0000
75%,2025-09-07 19:14:19.750000128,"7,030.9375",1.0000,12.0000,17.2400,17.0000
max,2025-12-15 13:59:36,"79,511.3000",3.0000,24.0000,764.2900,23.0000
std,NaN,"9,885.3087",0.2322,7.2024,24.6337,6.7615



tipo_operacion:


tipo_operacion
compra               319524
transf_entrada        92167
transf_salida         90116
cargo_recurrente      67249
pago_credito          51651
pago_servicio         48562
abono_inversion       41769
retiro_cajero         38989
deposito_oxxo         23152
cashback              14226
retiro_inversion      11822
deposito_farmacia      3157
Name: count, dtype: int64


estatus:


estatus
completada      748267
no_procesada     26609
en_disputa       19033
revertida         8475
Name: count, dtype: int64


Rango de fechas: 2025-01-01 08:00:09  ->  2025-12-15 13:59:36


In [ ]:
print("=== dataset_50k_anonymized ===")
print(f"Rango de fechas: {df_convs['date'].min()}  ->  {df_convs['date'].max()}")
print(f"\nchannel_source (1=texto, 2=voz):")
display(df_convs["channel_source"].value_counts())
turnos_por_conv = df_convs.groupby("conv_id").size()
print(f"\nTurnos por conversación: media={turnos_por_conv.mean():.2f}  mediana={turnos_por_conv.median():.0f}  max={turnos_por_conv.max()}")
display(turnos_por_conv.value_counts().sort_index().head(10))

=== dataset_50k_anonymized ===
Rango de fechas: 2025-01-07 03:16:22.067262795  ->  2025-11-28 20:02:33.620581593

channel_source (1=texto, 2=voz):


channel_source
1    46936
2     3063
Name: count, dtype: int64


Turnos por conversación: media=2.07  mediana=1  max=21


1     15000
3      4987
4      2329
5       979
6       438
7       189
8        85
9        43
10       25
11       16
Name: count, dtype: int64

## 6. Cardinalidades

In [ ]:
cardinalidad = pd.DataFrame([
    {"Tabla": "hey_clientes",           "Clave": "user_id",    "Únicos": df_clientes["user_id"].nunique()},
    {"Tabla": "hey_productos",          "Clave": "user_id",    "Únicos": df_productos["user_id"].nunique()},
    {"Tabla": "hey_productos",          "Clave": "producto_id","Únicos": df_productos["producto_id"].nunique()},
    {"Tabla": "hey_transacciones",      "Clave": "user_id",    "Únicos": df_transacc["user_id"].nunique()},
    {"Tabla": "hey_transacciones",      "Clave": "producto_id","Únicos": df_transacc["producto_id"].nunique()},
    {"Tabla": "dataset_50k_anonymized", "Clave": "user_id",    "Únicos": df_convs["user_id"].nunique()},
    {"Tabla": "dataset_50k_anonymized", "Clave": "conv_id",    "Únicos": df_convs["conv_id"].nunique()},
])
cardinalidad["Únicos"] = cardinalidad["Únicos"].map("{:,}".format)
display(cardinalidad)

,Tabla,Clave,Únicos
0,hey_clientes,user_id,"15,025"
1,hey_productos,user_id,"15,025"
2,hey_productos,producto_id,"38,909"
3,hey_transacciones,user_id,"15,025"
4,hey_transacciones,producto_id,"29,972"
5,dataset_50k_anonymized,user_id,"15,025"
6,dataset_50k_anonymized,conv_id,"24,119"


## 7. Validación de integridad referencial

In [ ]:
ids_c    = set(df_clientes["user_id"])
ids_p    = set(df_productos["user_id"])
ids_t    = set(df_transacc["user_id"])
ids_cv   = set(df_convs["user_id"])
ids_prod = set(df_productos["producto_id"])
ids_prod_t = set(df_transacc["producto_id"])

checks = [
    ("user_id en productos  → clientes",       len(ids_p  - ids_c),    0),
    ("user_id en transacc   → clientes",       len(ids_t  - ids_c),    0),
    ("user_id en convs      → clientes",       len(ids_cv - ids_c),    0),
    ("producto_id en transacc → productos",    len(ids_prod_t - ids_prod), 0),
]

print(f"{'Check':<45} {'Huérfanos':>12}  {'Estado':>8}")
print("-" * 70)
all_ok = True
for desc, huerfanos, esperado in checks:
    estado = "OK" if huerfanos == esperado else "FALLO"
    if estado == "FALLO":
        all_ok = False
    print(f"{desc:<45} {huerfanos:>12,}  {estado:>8}")

print()
print(f"Clientes sin ninguna conversacion: {len(ids_c - ids_cv):,}")
print(f"Productos sin ninguna transaccion: {len(ids_prod - ids_prod_t):,}")
print()
if all_ok:
    print("Integridad referencial: COMPLETA — no hay registros huerfanos.")
else:
    print("ADVERTENCIA: se encontraron registros huerfanos. Revisar arriba.")

Check                                            Huérfanos    Estado
----------------------------------------------------------------------
user_id en productos  → clientes                         0        OK
user_id en transacc   → clientes                         0        OK
user_id en convs      → clientes                         0        OK
producto_id en transacc → productos                      0        OK

Clientes sin ninguna conversacion: 0
Productos sin ninguna transaccion: 8,937

Integridad referencial: COMPLETA — no hay registros huerfanos.


In [ ]:
# Asserts formales — fallan si la integridad no es perfecta
assert df_productos["user_id"].isin(df_clientes["user_id"]).all(), \
    "hay user_id en productos no presentes en clientes"

assert df_transacc["user_id"].isin(df_clientes["user_id"]).all(), \
    "hay user_id en transacciones no presentes en clientes"

assert df_convs["user_id"].isin(df_clientes["user_id"]).all(), \
    "hay user_id en conversaciones no presentes en clientes"

assert df_transacc["producto_id"].isin(df_productos["producto_id"]).all(), \
    "hay producto_id en transacciones no presentes en productos"

print("Todos los asserts de integridad referencial pasaron correctamente.")

Todos los asserts de integridad referencial pasaron correctamente.


## 8. Joins entre tablas

### 8a. Clientes + Productos (`user_id`, left)

In [ ]:
clientes_productos = df_clientes.merge(
    df_productos,
    on="user_id",
    how="left",
    suffixes=("_cli", "_prod"),
)
print(f"Shape: {clientes_productos.shape}")
print(f"Promedio de productos por cliente: {len(df_productos) / len(df_clientes):.2f}")
display(clientes_productos.head(3))

Shape: (38909, 33)
Promedio de productos por cliente: 2.59


,user_id,edad,sexo,estado,ciudad,nivel_educativo,ocupacion,ingreso_mensual_mxn,antiguedad_dias,es_hey_pro,nomina_domiciliada,canal_apertura,score_buro,dias_desde_ultimo_login,preferencia_canal,satisfaccion_1_10,recibe_remesas,usa_hey_shop,idioma_preferido,tiene_seguro,num_productos_activos,patron_uso_atipico,producto_id,tipo_producto,fecha_apertura,estatus,limite_credito,saldo_actual,utilizacion_pct,tasa_interes_anual,plazo_meses,monto_mensualidad,fecha_ultimo_movimiento
0,USR-00001,21,M,Ciudad de México,CDMX - Benito Juárez,Preparatoria,Empleado,24500,1554,True,False,App,527,1,app_android,10.0000,False,True,es_MX,False,2,False,PRD-00000001,cuenta_debito,2023-06-26,activo,NaN,"80,954.6000",NaN,NaN,NaN,NaN,2025-11-27
1,USR-00001,21,M,Ciudad de México,CDMX - Benito Juárez,Preparatoria,Empleado,24500,1554,True,False,App,527,1,app_android,10.0000,False,True,es_MX,False,2,False,PRD-00000002,tarjeta_credito_hey,2022-10-16,activo,"144,000.0000","88,790.4000",0.6166,35.7100,NaN,NaN,2025-09-17
2,USR-00002,18,M,Jalisco,Puerto Vallarta,Preparatoria,Estudiante,19000,1410,True,False,App,714,3,app_android,8.0000,False,True,es_MX,True,2,False,PRD-00000003,cuenta_debito,2025-04-03,activo,NaN,"20,712.5400",NaN,NaN,NaN,NaN,2025-10-16


### 8b. Transacciones + Productos (`producto_id`, left)

In [ ]:
txn_productos = df_transacc.merge(
    df_productos[["producto_id", "user_id", "tipo_producto", "estatus",
                  "limite_credito", "saldo_actual", "utilizacion_pct"]],
    on="producto_id",
    how="left",
    suffixes=("_txn", "_prod"),
)
print(f"Shape: {txn_productos.shape}")
display(txn_productos.head(3))

Shape: (802384, 27)


,transaccion_id,user_id_txn,producto_id,fecha_hora,tipo_operacion,canal,monto,comercio_nombre,categoria_mcc,ciudad_transaccion,estatus_txn,motivo_no_procesada,intento_numero,meses_diferidos,cashback_generado,descripcion_libre,hora_del_dia,dia_semana,es_internacional,dispositivo,patron_uso_atipico,user_id_prod,tipo_producto,estatus_prod,limite_credito,saldo_actual,utilizacion_pct
0,TXN-0000000055,USR-00001,PRD-00000002,2025-01-15 14:17:42,compra,app_ios,33.8800,DivertidoPark,entretenimiento,"Nueva York, NY",completada,NaN,1,NaN,0.3400,Cargo automático,14,Wednesday,True,app_ios,False,USR-00001,tarjeta_credito_hey,activo,"144,000.0000","88,790.4000",0.6166
1,TXN-0000000048,USR-00001,PRD-00000001,2025-01-17 00:31:56,cargo_recurrente,app_ios,249.0000,GamerPass,servicios_digitales,CDMX - Benito Juárez,completada,NaN,1,NaN,NaN,Cargo automático,0,Friday,False,app_ios,False,USR-00001,cuenta_debito,activo,NaN,"80,954.6000",NaN
2,TXN-0000000018,USR-00001,PRD-00000001,2025-01-17 22:48:23,cargo_recurrente,app_huawei,399.0000,CloudDrive MX,servicios_digitales,CDMX - Benito Juárez,completada,NaN,1,NaN,NaN,inv hey,22,Friday,False,app_huawei,False,USR-00001,cuenta_debito,activo,NaN,"80,954.6000",NaN


### 8c. Transacciones + Productos + Clientes (join completo)

In [ ]:
df_full = (
    df_transacc
    .merge(
        df_productos[["producto_id", "tipo_producto", "estatus",
                      "limite_credito", "saldo_actual", "utilizacion_pct",
                      "tasa_interes_anual"]],
        on="producto_id",
        how="left",
        suffixes=("", "_prod"),
    )
    .merge(
        df_clientes,
        on="user_id",
        how="left",
        suffixes=("_txn", "_cli"),
    )
)
print(f"Shape join completo: {df_full.shape}")
display(df_full.head(3))

Shape join completo: (802384, 48)


,transaccion_id,user_id,producto_id,fecha_hora,tipo_operacion,canal,monto,comercio_nombre,categoria_mcc,ciudad_transaccion,estatus,motivo_no_procesada,intento_numero,meses_diferidos,cashback_generado,descripcion_libre,hora_del_dia,dia_semana,es_internacional,dispositivo,patron_uso_atipico_txn,tipo_producto,estatus_prod,limite_credito,saldo_actual,utilizacion_pct,tasa_interes_anual,edad,sexo,estado,ciudad,nivel_educativo,ocupacion,ingreso_mensual_mxn,antiguedad_dias,es_hey_pro,nomina_domiciliada,canal_apertura,score_buro,dias_desde_ultimo_login,preferencia_canal,satisfaccion_1_10,recibe_remesas,usa_hey_shop,idioma_preferido,tiene_seguro,num_productos_activos,patron_uso_atipico_cli
0,TXN-0000000055,USR-00001,PRD-00000002,2025-01-15 14:17:42,compra,app_ios,33.8800,DivertidoPark,entretenimiento,"Nueva York, NY",completada,NaN,1,NaN,0.3400,Cargo automático,14,Wednesday,True,app_ios,False,tarjeta_credito_hey,activo,"144,000.0000","88,790.4000",0.6166,35.7100,21,M,Ciudad de México,CDMX - Benito Juárez,Preparatoria,Empleado,24500,1554,True,False,App,527,1,app_android,10.0000,False,True,es_MX,False,2,False
1,TXN-0000000048,USR-00001,PRD-00000001,2025-01-17 00:31:56,cargo_recurrente,app_ios,249.0000,GamerPass,servicios_digitales,CDMX - Benito Juárez,completada,NaN,1,NaN,NaN,Cargo automático,0,Friday,False,app_ios,False,cuenta_debito,activo,NaN,"80,954.6000",NaN,NaN,21,M,Ciudad de México,CDMX - Benito Juárez,Preparatoria,Empleado,24500,1554,True,False,App,527,1,app_android,10.0000,False,True,es_MX,False,2,False
2,TXN-0000000018,USR-00001,PRD-00000001,2025-01-17 22:48:23,cargo_recurrente,app_huawei,399.0000,CloudDrive MX,servicios_digitales,CDMX - Benito Juárez,completada,NaN,1,NaN,NaN,inv hey,22,Friday,False,app_huawei,False,cuenta_debito,activo,NaN,"80,954.6000",NaN,NaN,21,M,Ciudad de México,CDMX - Benito Juárez,Preparatoria,Empleado,24500,1554,True,False,App,527,1,app_android,10.0000,False,True,es_MX,False,2,False


### 8d. Conversaciones + Clientes (`user_id`, left)

In [ ]:
convs_clientes = df_convs.merge(
    df_clientes[["user_id", "edad", "sexo", "estado", "ocupacion",
                 "ingreso_mensual_mxn", "es_hey_pro", "satisfaccion_1_10",
                 "num_productos_activos"]],
    on="user_id",
    how="left",
)
print(f"Shape conversaciones + clientes: {convs_clientes.shape}")
print(f"Nulos en columnas de clientes tras join: {convs_clientes[['edad','sexo']].isnull().sum().to_dict()}")
display(convs_clientes.head(3))

Shape conversaciones + clientes: (49999, 14)
Nulos en columnas de clientes tras join: {'edad': 0, 'sexo': 0}


,input,output,date,conv_id,user_id,channel_source,edad,sexo,estado,ocupacion,ingreso_mensual_mxn,es_hey_pro,satisfaccion_1_10,num_productos_activos
0,"Me enteré de una promo ""Supercashback Pagos Gubernamenta...",Claro que puedo ayudarte! Para participar en la promoció...,2025-08-17,000502c2-288c-41f6-b751-a8b45a376a81,USR-09092,1,32,H,Tamaulipas,Empleado,33500,True,NaN,2
1,La tarjeta de crédito Hey Negocios es diferente de la TD...,Claro! La Tarjeta de Crédito Hey Negocios es diferente d...,2025-08-17,000502c2-288c-41f6-b751-a8b45a376a81,USR-09092,1,32,H,Tamaulipas,Empleado,33500,True,NaN,2
2,"Entiendo, gracias",¡De nada! Me alegra haber podido ayudarte. Si tienes más...,2025-08-17,000502c2-288c-41f6-b751-a8b45a376a81,USR-09092,1,32,H,Tamaulipas,Empleado,33500,True,NaN,2


## 9. Resumen ejecutivo

In [ ]:
resumen = pd.DataFrame([
    {
        "Join": "clientes ← productos (user_id)",
        "Tipo": "left",
        "Filas": f"{clientes_productos.shape[0]:,}",
        "Cols": clientes_productos.shape[1],
        "Nulos post-join": 0,
    },
    {
        "Join": "transacciones ← productos (producto_id)",
        "Tipo": "left",
        "Filas": f"{txn_productos.shape[0]:,}",
        "Cols": txn_productos.shape[1],
        "Nulos post-join": int(txn_productos["tipo_producto"].isnull().sum()),
    },
    {
        "Join": "transacciones ← productos ← clientes",
        "Tipo": "left+left",
        "Filas": f"{df_full.shape[0]:,}",
        "Cols": df_full.shape[1],
        "Nulos post-join": int(df_full["edad"].isnull().sum()),
    },
    {
        "Join": "conversaciones ← clientes (user_id)",
        "Tipo": "left",
        "Filas": f"{convs_clientes.shape[0]:,}",
        "Cols": convs_clientes.shape[1],
        "Nulos post-join": int(convs_clientes["edad"].isnull().sum()),
    },
])

display(resumen)
print()
print("Integridad referencial: 15,025 user_id coinciden en las 4 tablas. Sin huerfanos.")
print("Datasets listos para analisis.")

,Join,Tipo,Filas,Cols,Nulos post-join
0,clientes ← productos (user_id),left,"38,909",33,0
1,transacciones ← productos (producto_id),left,"802,384",27,0
2,transacciones ← productos ← clientes,left+left,"802,384",48,0
3,conversaciones ← clientes (user_id),left,"49,999",14,0



Integridad referencial: 15,025 user_id coinciden en las 4 tablas. Sin huerfanos.
Datasets listos para analisis.
